In [ ]:
# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_excel("/content/WB Dataset Merged.xlsx")

In [ ]:
df.head()

,Identifier,General (=1 if General),Source,Country,Reg,Inc,Historical Inc,Year,ISO,General Notes,...,Unnamed: 47,Taxes on International Trade,Unnamed: 49,Unnamed: 50,Other Taxes,Non-Tax Revenue,Unnamed: 53,Unnamed: 54,Social Contributions,Grants
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,o/w Excises,Total,o/w Import,o/w Export,NaN,Total,Resource Component,Non-Resource Component,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ABW1980,0.0,NaN,NaN,NaN,NaN,NaN,1980.0,ABW,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ABW1981,0.0,NaN,NaN,NaN,NaN,NaN,1981.0,ABW,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABW1982,0.0,NaN,NaN,NaN,NaN,NaN,1982.0,ABW,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import pandas as pd

# Load the data, skipping the triple-header rows
df_raw = pd.read_excel('/content/WB Dataset Merged.xlsx', skiprows=3, header=None)

# CORRECTED Column Mapping
# Key fix: cols 46-50 were all shifted by one position in the original code.
# Col 45 = General Sales Tax (broad measure, includes VAT+turnover taxes)
# Col 46 = VAT specifically           ← was wrongly named EXCISE_Tax before
# Col 47 = Excise taxes               ← was wrongly named TRADE_Tax_Total before
# Col 48 = Trade taxes total          ← was wrongly named TRADE_Import before
# Col 49 = Import duties              ← was wrongly named TRADE_Export before
# Col 50 = Export duties              ← was MISSING entirely from original code

final_mapping = {
    8:  'ID_ISO',
    3:  'ID_Country',
    7:  'ID_Year',
    4:  'ID_Region',
    5:  'INC_Status',
    6:  'INC_Historical',
    17: 'REV_Total_Inc_Grants',
    19: 'REV_Total_Exc_Grants',
    21: 'REV_Resource_Total',
    22: 'REV_NonResource_Total',
    23: 'TAX_Total_Inc_SC',
    24: 'TAX_Total_Exc_SC',
    25: 'TAX_Resource_Total',
    26: 'TAX_NonResource_Total',
    28: 'DIR_Tax_Inc_SC',
    30: 'DIR_Tax_Exc_SC',
    32: 'INC_Tax_Total',
    34: 'INC_Tax_NonResource',
    35: 'PIT_Total',
    36: 'CIT_Total',
    37: 'CIT_Resource',
    38: 'CIT_NonResource',
    39: 'PAYROLL_Tax',
    40: 'PROPERTY_Tax',
    41: 'IND_Tax_Total',
    45: 'GST_General',       # Broad General Sales Tax (VAT + turnover taxes)
    46: 'VAT_Total',         # CORRECTED: pure VAT — this is your Tax Mix variable
    47: 'EXCISE_Tax',        # CORRECTED: excise duties
    48: 'TRADE_Tax_Total',   # CORRECTED: total international trade taxes
    49: 'TRADE_Import',      # CORRECTED: import duties
    50: 'TRADE_Export',      # ADDED: export duties (was missing entirely)
    51: 'OTHER_Taxes',
    52: 'NON_Tax_Revenue'
}

# Apply selection and naming
df_final = df_raw[list(final_mapping.keys())].copy()
df_final.columns = list(final_mapping.values())

# Convert all revenue/tax columns to float
numeric_cols = [c for c in df_final.columns if 'ID_' not in c and 'INC_' not in c]
for col in numeric_cols:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

# Quick sanity check on VAT_Total for known countries
print("=== Sanity check: VAT_Total for known high-VAT countries ===")
print("(Expected: Germany ~7%, UK ~7%, USA near 0 at federal level)")
for iso in ['DEU', 'GBR', 'USA']:
    subset = df_final[df_final['ID_ISO'] == iso]['VAT_Total'].dropna()
    if not subset.empty:
        print(f"  {iso}: mean = {subset.mean():.4f}  |  latest = {subset.iloc[-1]:.4f}")

print(f"\nFinal Version Structured: All subsections are now single columns.")
print(f"Total Columns: {len(df_final.columns)}")
df_final.head()

=== Sanity check: VAT_Total for known high-VAT countries ===
(Expected: Germany ~7%, UK ~7%, USA near 0 at federal level)
  DEU: mean = 0.0640  |  latest = 0.0724
  GBR: mean = 0.0584  |  latest = 0.0692
  USA: mean = 0.0000  |  latest = 0.0000

Final Version Structured: All subsections are now single columns.
Total Columns: 33


,ID_ISO,ID_Country,ID_Year,ID_Region,INC_Status,INC_Historical,REV_Total_Inc_Grants,REV_Total_Exc_Grants,REV_Resource_Total,REV_NonResource_Total,...,PROPERTY_Tax,IND_Tax_Total,GST_General,VAT_Total,EXCISE_Tax,TRADE_Tax_Total,TRADE_Import,TRADE_Export,OTHER_Taxes,NON_Tax_Revenue
0,ABW,NaN,1980,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ABW,NaN,1981,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ABW,NaN,1982,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ABW,NaN,1983,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABW,NaN,1984,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 1. Final check of the columns you currently have
print("Columns currently in df_final (Total: {}):".format(len(df_final.columns)))
print(df_final.columns.tolist())

# 2. Column reference note:
# TRADE_Tax_Total, TRADE_Import, TRADE_Export, GST_General are kept in the
# saved file for completeness but will be EXCLUDED from model predictors
# during the analytical stage, as per the proposal methodology.

# 3. Save the structured dataset to a CSV file for download
df_final.to_csv('Cleaned_Tax_Dataset.csv', index=False)

# 4. Save to Excel
df_final.to_excel('Cleaned_Tax_Dataset.xlsx', index=False)

print("\nSuccess! The cleaned dataset has been saved.")
print("You can now find 'Cleaned_Tax_Dataset.csv' in the Colab file explorer on the left.")

Columns currently in df_final (Total: 33):
['ID_ISO', 'ID_Country', 'ID_Year', 'ID_Region', 'INC_Status', 'INC_Historical', 'REV_Total_Inc_Grants', 'REV_Total_Exc_Grants', 'REV_Resource_Total', 'REV_NonResource_Total', 'TAX_Total_Inc_SC', 'TAX_Total_Exc_SC', 'TAX_Resource_Total', 'TAX_NonResource_Total', 'DIR_Tax_Inc_SC', 'DIR_Tax_Exc_SC', 'INC_Tax_Total', 'INC_Tax_NonResource', 'PIT_Total', 'CIT_Total', 'CIT_Resource', 'CIT_NonResource', 'PAYROLL_Tax', 'PROPERTY_Tax', 'IND_Tax_Total', 'GST_General', 'VAT_Total', 'EXCISE_Tax', 'TRADE_Tax_Total', 'TRADE_Import', 'TRADE_Export', 'OTHER_Taxes', 'NON_Tax_Revenue']

Success! The cleaned dataset has been saved.
You can now find 'Cleaned_Tax_Dataset.csv' in the Colab file explorer on the left.


In [ ]:
# ============================================================
# STEP 3: Add Entrepreneurial Entry Rate
# Simple approach — calculate density from what we have
# Density = (New firms registered / Working-age population) × 1000
# ============================================================

!pip install wbgapi
import wbgapi as wb
import pandas as pd
import numpy as np

# ── Load the cleaned tax dataset ─────────────────────────────
df = pd.read_excel('/content/Cleaned_Tax_Dataset.xlsx')
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")

# ── Fetch new business registrations ─────────────────────────
print("\nFetching new business registrations (IC.BUS.NREG)...")
raw_count = wb.data.DataFrame('IC.BUS.NREG', time=range(1980, 2023), labels=False)
raw_count = raw_count.reset_index().rename(columns={'economy': 'ID_ISO'})
year_cols = [c for c in raw_count.columns if str(c).startswith('YR')]
df_count = raw_count.melt(id_vars='ID_ISO', value_vars=year_cols,
                          var_name='ID_Year', value_name='ENTRY_Count')
df_count['ID_Year'] = df_count['ID_Year'].str.replace('YR', '').astype(int)
df_count['ENTRY_Count'] = pd.to_numeric(df_count['ENTRY_Count'], errors='coerce')
print(f"  ✓ {df_count['ENTRY_Count'].notna().sum()} observations retrieved")

# ── Fetch working-age population (ages 15–64) ─────────────────
print("\nFetching working-age population (SP.POP.1564.TO)...")
raw_pop = wb.data.DataFrame('SP.POP.1564.TO', time=range(1980, 2023), labels=False)
raw_pop = raw_pop.reset_index().rename(columns={'economy': 'ID_ISO'})
year_cols = [c for c in raw_pop.columns if str(c).startswith('YR')]
df_pop = raw_pop.melt(id_vars='ID_ISO', value_vars=year_cols,
                      var_name='ID_Year', value_name='POP_WorkingAge')
df_pop['ID_Year'] = df_pop['ID_Year'].str.replace('YR', '').astype(int)
df_pop['POP_WorkingAge'] = pd.to_numeric(df_pop['POP_WorkingAge'], errors='coerce')
print(f"  ✓ {df_pop['POP_WorkingAge'].notna().sum()} observations retrieved")

# ── Merge both into the tax dataset ──────────────────────────
df = df.merge(df_count, on=['ID_ISO', 'ID_Year'], how='left')
df = df.merge(df_pop,   on=['ID_ISO', 'ID_Year'], how='left')

# ── Calculate entry density ───────────────────────────────────
# Formula: (new firms / working-age population) × 1,000
# This is the exact World Bank definition of IC.BUS.DFRN.ZS
df['ENTRY_Density'] = (df['ENTRY_Count'] / df['POP_WorkingAge']) * 1000
print(f"\n✓ ENTRY_Density calculated: "
      f"{df['ENTRY_Density'].notna().sum()} non-null rows")

# ── Coverage report ───────────────────────────────────────────
print("\n" + "="*55)
print("DEPENDENT VARIABLE COVERAGE REPORT")
print("="*55)
for col, label in [('ENTRY_Count','Raw count'), ('ENTRY_Density','Density (per 1k)')]:
    n       = df[col].notna().sum()
    pct     = n / len(df) * 100
    ctries  = df.dropna(subset=[col])['ID_ISO'].nunique()
    yr_min  = df.dropna(subset=[col])['ID_Year'].min()
    yr_max  = df.dropna(subset=[col])['ID_Year'].max()
    print(f"\n{label} ({col}):")
    print(f"  Observations  : {n:,} / {len(df):,} ({pct:.1f}%)")
    print(f"  Countries     : {ctries}")
    print(f"  Years covered : {yr_min} – {yr_max}")

# ── Sanity check — density should be small positive numbers ──
print("\n=== Density sanity check (2018–2021) ===")
check = df[df['ID_ISO'].isin(['USA','GBR','DEU','IND','KEN']) &
           df['ID_Year'].between(2018, 2021)]\
        [['ID_ISO','ID_Year','ENTRY_Count','POP_WorkingAge','ENTRY_Density']]\
        .dropna().sort_values(['ID_ISO','ID_Year'])
print(check.to_string(index=False))

# ── Save ──────────────────────────────────────────────────────
df.to_csv('Merged_Tax_Entry_Dataset.csv', index=False)
df.to_excel('Merged_Tax_Entry_Dataset.xlsx', index=False)
print(f"\n✓ Final dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print("✓ Saved: Merged_Tax_Entry_Dataset.csv and Merged_Tax_Entry_Dataset.xlsx")

Loaded: 8408 rows × 33 columns

Fetching new business registrations (IC.BUS.NREG)...
  ✓ 2370 observations retrieved

Fetching working-age population (SP.POP.1564.TO)...
  ✓ 11385 observations retrieved

✓ ENTRY_Density calculated: 2312 non-null rows

DEPENDENT VARIABLE COVERAGE REPORT

Raw count (ENTRY_Count):
  Observations  : 2,312 / 8,408 (27.5%)
  Countries     : 170
  Years covered : 2006 – 2022

Density (per 1k) (ENTRY_Density):
  Observations  : 2,312 / 8,408 (27.5%)
  Countries     : 170
  Years covered : 2006 – 2022

=== Density sanity check (2018–2021) ===
ID_ISO  ID_Year  ENTRY_Count  POP_WorkingAge  ENTRY_Density
   DEU     2018      72844.0      53878496.0       1.352005
   DEU     2019      73991.0      53768633.0       1.376100
   DEU     2020      72774.0      53550233.0       1.358986
   DEU     2021      82991.0      53251922.0       1.558460
   GBR     2018     664974.0      42221097.0      15.749804
   GBR     2019     684874.0      42323933.0      16.181719
   GBR

In [ ]:
import pandas as pd

df = pd.read_excel('/content/Merged_Tax_Entry_Dataset.xlsx')
df.isnull().sum()

,0
ID_ISO,0
ID_Country,1055
ID_Year,0
ID_Region,1055
INC_Status,1055
INC_Historical,2041
REV_Total_Inc_Grants,2548
REV_Total_Exc_Grants,2104
REV_Resource_Total,3417
REV_NonResource_Total,3591


In [ ]:
# ============================================================
# STEP 4A: DATA TYPE & QUALITY FIX
# Run this BEFORE the imputation code
# ============================================================

import pandas as pd
import numpy as np

df = pd.read_excel('/content/Merged_Tax_Entry_Dataset.xlsx')
df = df.sort_values(['ID_ISO', 'ID_Year']).reset_index(drop=True)
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")

#+ Fix 1: Clip negative values to zero
# Tax revenue as % of GDP cannot be negative.
# These are World Bank source recording artifacts.
clip_cols = [
    'REV_Resource_Total', 'TAX_Resource_Total',
    'CIT_Resource', 'PAYROLL_Tax', 'PROPERTY_Tax',
    'TRADE_Tax_Total', 'TRADE_Import', 'TRADE_Export',
    'OTHER_Taxes', 'NON_Tax_Revenue'
]

print("\n--- Clipping negative values to zero ---")
for col in clip_cols:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        df[col] = df[col].clip(lower=0)
        print(f"  {col:<28}: {neg_count} negative values clipped to 0")

# Verify no negatives remain
remaining_neg = sum((df[col] < 0).sum() for col in clip_cols)
print(f"\n  ✓ Negative values remaining: {remaining_neg}")

# ── Fix 2: ID_Country — stays as object (string), correct ────
# NaN values here will be handled in Group 1 imputation below
print(f"\n  ID_Country dtype: {df['ID_Country'].dtype} ✓")

# ── Note: ID_Region, INC_Status, INC_Historical ──────────────
# These stay as float64 FOR NOW — NaN cannot live in int columns.
# They will be converted to int AFTER imputation fills the gaps.
print(f"  ID_Region dtype     : {df['ID_Region'].dtype} (will convert to int after imputation)")
print(f"  INC_Status dtype    : {df['INC_Status'].dtype} (will convert to int after imputation)")
print(f"  INC_Historical dtype: {df['INC_Historical'].dtype} (will convert to int after imputation)")

# ── Final check ───────────────────────────────────────────────
print(f"\n✓ Dataset ready for imputation: {df.shape[0]} rows × {df.shape[1]} columns")

# Save the fixed version — this becomes the input for imputation
df.to_excel('Fixed_Tax_Entry_Dataset.xlsx', index=False)
df.to_csv('Fixed_Tax_Entry_Dataset.csv', index=False)
print("✓ Saved: Fixed_Tax_Entry_Dataset.xlsx and Fixed_Tax_Entry_Dataset.csv")

Loaded: 8408 rows × 36 columns

--- Clipping negative values to zero ---
  REV_Resource_Total          : 10 negative values clipped to 0
  TAX_Resource_Total          : 9 negative values clipped to 0
  CIT_Resource                : 8 negative values clipped to 0
  PAYROLL_Tax                 : 5 negative values clipped to 0
  PROPERTY_Tax                : 3 negative values clipped to 0
  TRADE_Tax_Total             : 15 negative values clipped to 0
  TRADE_Import                : 1 negative values clipped to 0
  TRADE_Export                : 5 negative values clipped to 0
  OTHER_Taxes                 : 45 negative values clipped to 0
  NON_Tax_Revenue             : 2 negative values clipped to 0

  ✓ Negative values remaining: 0

  ID_Country dtype: object ✓
  ID_Region dtype     : float64 (will convert to int after imputation)
  INC_Status dtype    : float64 (will convert to int after imputation)
  INC_Historical dtype: float64 (will convert to int after imputation)

✓ Dataset ready 

In [ ]:
# ============================================================
# STEP 4: MISSING VALUE IMPUTATION
# Hierarchical Panel Imputation — 5 groups + DV protection
# Input:  Merged_Tax_Entry_Dataset.xlsx
# Output: Imputed_Tax_Dataset.csv / .xlsx
# ============================================================

import pandas as pd
import numpy as np

df = pd.read_excel('/content/Fixed_Tax_Entry_Dataset.xlsx')
df = df.sort_values(['ID_ISO', 'ID_Year']).reset_index(drop=True)
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")

# ── Protect dependent variables — never impute these ─────────
DV_COLS = ['ENTRY_Count', 'ENTRY_Density']
print(f"\n✓ Protected (will NOT be imputed): {DV_COLS}")

# ════════════════════════════════════════════════════════════
# GROUP 1 — Structural identifiers
# ID_Country, ID_Region, INC_Status
# Same value every year per country — just absent in early rows
# ════════════════════════════════════════════════════════════
print("\n--- Group 1: Structural identifiers ---")
for col in ['ID_Country', 'ID_Region', 'INC_Status']:
    before = df[col].isnull().sum()
    df[col] = df.groupby('ID_ISO')[col].transform(lambda s: s.ffill().bfill())
    after  = df[col].isnull().sum()
    print(f"  {col:<20}: {before:>4} → {after} missing")

# ════════════════════════════════════════════════════════════
# GROUP 2 — Income classification target
# INC_Historical: MAR — preserves real income group transitions
# ════════════════════════════════════════════════════════════
print("\n--- Group 2: INC_Historical ---")
before = df['INC_Historical'].isnull().sum()
df['INC_Historical'] = df.groupby('ID_ISO')['INC_Historical'].transform(
    lambda s: s.ffill().bfill()
)
print(f"  INC_Historical      : {before:>4} → {df['INC_Historical'].isnull().sum()} missing")

# ════════════════════════════════════════════════════════════
# GROUP 3 — Core tax predictors
# Strategy: 4-step cascade
#   Step 1: Linear interpolation within country × time
#   Step 2: Forward + backward fill (handles edge years)
#   Step 3: Region × IncomeGroup × Year median (peer benchmark)
#   Step 4: Global year median (final fallback)
# Binary _WAS_MISSING flags saved BEFORE any imputation
# ════════════════════════════════════════════════════════════
print("\n--- Group 3: Core tax predictors (4-step cascade) ---")

core_tax_cols = [
    'PIT_Total', 'CIT_Total', 'PAYROLL_Tax', 'PROPERTY_Tax',
    'VAT_Total', 'GST_General', 'IND_Tax_Total', 'EXCISE_Tax',
    'OTHER_Taxes', 'NON_Tax_Revenue',
    'DIR_Tax_Inc_SC', 'DIR_Tax_Exc_SC',
    'INC_Tax_Total', 'INC_Tax_NonResource',
    'TAX_Total_Inc_SC', 'TAX_Total_Exc_SC', 'TAX_NonResource_Total',
    'REV_Total_Inc_Grants', 'REV_Total_Exc_Grants', 'REV_NonResource_Total'
]

# Missingness flags for the 6 key analytical columns only
flag_cols = ['PIT_Total', 'CIT_Total', 'PAYROLL_Tax',
             'PROPERTY_Tax', 'VAT_Total', 'IND_Tax_Total']
print("\n  Creating missingness indicator flags...")
for col in flag_cols:
    df[col + '_WAS_MISSING'] = df[col].isnull().astype(int)
    print(f"    {col}_WAS_MISSING flagged: {df[col].isnull().sum()} rows")

# Step 1: Linear interpolation within each country's time series
print("\n  Step 1 — Linear interpolation within country...")
for col in core_tax_cols:
    before = df[col].isnull().sum()
    df[col] = df.groupby('ID_ISO')[col].transform(
        lambda s: s.interpolate(method='linear', limit_direction='both')
    )
    after = df[col].isnull().sum()
    if before > after:
        print(f"    {col:<28}: {before:>4} → {after}")

# Step 2: Forward + backward fill for edge years
print("\n  Step 2 — Forward + backward fill...")
for col in core_tax_cols:
    before = df[col].isnull().sum()
    df[col] = df.groupby('ID_ISO')[col].transform(lambda s: s.ffill().bfill())
    after = df[col].isnull().sum()
    if before > after:
        print(f"    {col:<28}: {before:>4} → {after}")

# Step 3: Region × IncomeGroup × Year median
print("\n  Step 3 — Region × income group × year median...")
for col in core_tax_cols:
    before = df[col].isnull().sum()
    if before == 0:
        continue
    peer_med = df.groupby(
        ['ID_Region', 'INC_Historical', 'ID_Year']
    )[col].transform('median')
    df[col] = df[col].fillna(peer_med)
    after = df[col].isnull().sum()
    if before > after:
        print(f"    {col:<28}: {before:>4} → {after}")

# Step 4: Global year median — last resort
print("\n  Step 4 — Global year median fallback...")
for col in core_tax_cols:
    before = df[col].isnull().sum()
    if before == 0:
        continue
    global_med = df.groupby('ID_Year')[col].transform('median')
    df[col] = df[col].fillna(global_med)
    after = df[col].isnull().sum()
    if before > after:
        print(f"    {col:<28}: {before:>4} → {after}")

# ════════════════════════════════════════════════════════════
# GROUP 4 — Structurally sparse (MNAR)
# CIT_Resource, CIT_NonResource, Resource totals, Trade taxes
# Countries with no reporting history → structural zero
# Countries with partial data → 4-step cascade
# ════════════════════════════════════════════════════════════
print("\n--- Group 4: MNAR structural variables ---")
mnar_cols = [
    'CIT_Resource', 'CIT_NonResource',
    'REV_Resource_Total', 'TAX_Resource_Total',
    'TRADE_Tax_Total', 'TRADE_Import', 'TRADE_Export'
]

for col in mnar_cols:
    before = df[col].isnull().sum()
    # Identify countries that never reported any non-zero value
    has_data = df.groupby('ID_ISO')[col].apply(lambda s: (s.dropna() > 0).any())
    no_data_isos = has_data[~has_data].index
    df.loc[df['ID_ISO'].isin(no_data_isos) & df[col].isnull(), col] = 0.0
    # Apply cascade for the rest
    df[col] = df.groupby('ID_ISO')[col].transform(
        lambda s: s.interpolate(method='linear', limit_direction='both')
    )
    df[col] = df.groupby('ID_ISO')[col].transform(lambda s: s.ffill().bfill())
    peer_med = df.groupby(
        ['ID_Region', 'INC_Historical', 'ID_Year']
    )[col].transform('median')
    df[col] = df[col].fillna(peer_med)
    global_med = df.groupby('ID_Year')[col].transform('median')
    df[col] = df[col].fillna(global_med)
    after = df[col].isnull().sum()
    print(f"  {col:<28}: {before:>4} → {after}")

# ════════════════════════════════════════════════════════════
# GROUP 5 — Working-age population (only 129 missing = 1.5%)
# Simple interpolation within country is sufficient
# ════════════════════════════════════════════════════════════
print("\n--- Group 5: POP_WorkingAge ---")
before = df['POP_WorkingAge'].isnull().sum()
df['POP_WorkingAge'] = df.groupby('ID_ISO')['POP_WorkingAge'].transform(
    lambda s: s.interpolate(method='linear', limit_direction='both')
)
df['POP_WorkingAge'] = df.groupby('ID_ISO')['POP_WorkingAge'].transform(
    lambda s: s.ffill().bfill()
)
print(f"  POP_WorkingAge      : {before:>4} → {df['POP_WorkingAge'].isnull().sum()} missing")

# ════════════════════════════════════════════════════════════
# FINAL VALIDATION REPORT
# ════════════════════════════════════════════════════════════
print("\n" + "="*55)
print("FINAL MISSING VALUE REPORT")
print("="*55)
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]

if len(remaining) == 0:
    print("✓ Zero missing values in all imputed columns.")
else:
    print("Remaining missing values:")
    for col, n in remaining.items():
        pct = n / len(df) * 100
        note = " ← DV: expected, do not impute" if col in DV_COLS else ""
        print(f"  {col:<28}: {n:>5} ({pct:.1f}%){note}")

print(f"\nFinal shape : {df.shape[0]} rows × {df.shape[1]} columns")
flag_added = [c for c in df.columns if '_WAS_MISSING' in c]
print(f"Flags added : {flag_added}")

# Save
df.to_csv('Imputed_Tax_Dataset.csv', index=False)
df.to_excel('Imputed_Tax_Dataset.xlsx', index=False)
print("\n✓ Saved: Imputed_Tax_Dataset.csv and Imputed_Tax_Dataset.xlsx")

Loaded: 8408 rows × 36 columns

✓ Protected (will NOT be imputed): ['ENTRY_Count', 'ENTRY_Density']

--- Group 1: Structural identifiers ---
  ID_Country          : 1055 → 0 missing
  ID_Region           : 1055 → 0 missing
  INC_Status          : 1055 → 0 missing

--- Group 2: INC_Historical ---
  INC_Historical      : 2041 → 43 missing

--- Group 3: Core tax predictors (4-step cascade) ---

  Creating missingness indicator flags...
    PIT_Total_WAS_MISSING flagged: 3561 rows
    CIT_Total_WAS_MISSING flagged: 3510 rows
    PAYROLL_Tax_WAS_MISSING flagged: 3889 rows
    PROPERTY_Tax_WAS_MISSING flagged: 3465 rows
    VAT_Total_WAS_MISSING flagged: 4541 rows
    IND_Tax_Total_WAS_MISSING flagged: 2243 rows

  Step 1 — Linear interpolation within country...
    PIT_Total                   : 3561 → 645
    CIT_Total                   : 3510 → 688
    PAYROLL_Tax                 : 3889 → 1485
    PROPERTY_Tax                : 3465 → 860
    VAT_Total                   : 4541 → 1548
    GS

In [ ]:
# ============================================================
# STEP 4B: CLEANUP — Fix 2 remaining incomplete imputations
# INC_Historical (43 missing) + POP_WorkingAge (129 missing)
# ============================================================

import pandas as pd
import numpy as np

df = pd.read_excel('/content/Imputed_Tax_Dataset.xlsx')
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")

print(f"\nBefore fix:")
print(f"  INC_Historical missing : {df['INC_Historical'].isnull().sum()}")
print(f"  POP_WorkingAge missing : {df['POP_WorkingAge'].isnull().sum()}")

# ── Fix 1: INC_Historical — regional median fallback ─────────
# These 43 rows belong to countries with zero historical
# income classification on record. Use the most common
# income group within their region as the fallback.
region_mode = df.groupby('ID_Region')['INC_Historical'].transform(
    lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan
)
df['INC_Historical'] = df['INC_Historical'].fillna(region_mode)

# Final fallback: global mode (catches any region also missing)
global_mode = df['INC_Historical'].mode().iloc[0]
df['INC_Historical'] = df['INC_Historical'].fillna(global_mode)

# Now safe to convert to integer
df['INC_Historical'] = df['INC_Historical'].astype(int)
df['INC_Status']     = df['INC_Status'].astype(int)
df['ID_Region']      = df['ID_Region'].astype(int)

print(f"\nAfter Fix 1 — INC_Historical: {df['INC_Historical'].isnull().sum()} missing")
print(f"  INC_Historical dtype now : {df['INC_Historical'].dtype} ✓")
print(f"  INC_Status dtype now     : {df['INC_Status'].dtype} ✓")
print(f"  ID_Region dtype now      : {df['ID_Region'].dtype} ✓")

# ── Fix 2: POP_WorkingAge — regional year median fallback ────
# These 129 rows are micro-nations with no WB population data.
# Use the median of their region × year peer group.
region_year_med = df.groupby(
    ['ID_Region', 'ID_Year']
)['POP_WorkingAge'].transform('median')
df['POP_WorkingAge'] = df['POP_WorkingAge'].fillna(region_year_med)

# Recalculate ENTRY_Density for any rows where POP was just fixed
# and ENTRY_Count exists
mask = (df['ENTRY_Count'].notna() &
        df['POP_WorkingAge'].notna() &
        df['ENTRY_Density'].isna())
df.loc[mask, 'ENTRY_Density'] = (
    df.loc[mask, 'ENTRY_Count'] / df.loc[mask, 'POP_WorkingAge']
) * 1000

print(f"\nAfter Fix 2 — POP_WorkingAge: {df['POP_WorkingAge'].isnull().sum()} missing")

# ── Final validation ──────────────────────────────────────────
print("\n" + "="*55)
print("FINAL CLEAN DATASET — MISSING VALUE REPORT")
print("="*55)
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]

DV_COLS = ['ENTRY_Count', 'ENTRY_Density']
if len(remaining) == 0:
    print("✓ Zero missing values across all columns.")
else:
    for col, n in remaining.items():
        pct  = n / len(df) * 100
        note = " ← DV: intentional" if col in DV_COLS else " ← UNEXPECTED"
        print(f"  {col:<28}: {n:>5} ({pct:.1f}%){note}")

print(f"\nFinal shape  : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"INC_Historical values : {sorted(df['INC_Historical'].unique())}")
print(f"ID_Region values      : {sorted(df['ID_Region'].unique())}")

# ── Save final clean dataset ──────────────────────────────────
df.to_csv('Final_Clean_Dataset.csv', index=False)
df.to_excel('Final_Clean_Dataset.xlsx', index=False)
print("\n✓ Saved: Final_Clean_Dataset.csv and Final_Clean_Dataset.xlsx")
print("  This is your master dataset for all analysis going forward.")

Loaded: 8408 rows × 42 columns

Before fix:
  INC_Historical missing : 43
  POP_WorkingAge missing : 129

After Fix 1 — INC_Historical: 0 missing
  INC_Historical dtype now : int64 ✓
  INC_Status dtype now     : int64 ✓
  ID_Region dtype now      : int64 ✓

After Fix 2 — POP_WorkingAge: 0 missing

FINAL CLEAN DATASET — MISSING VALUE REPORT
  ENTRY_Count                 :  6096 (72.5%) ← DV: intentional
  ENTRY_Density               :  6096 (72.5%) ← DV: intentional

Final shape  : 8408 rows × 42 columns
INC_Historical values : [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
ID_Region values      : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]

✓ Saved: Final_Clean_Dataset.csv and Final_Clean_Dataset.xlsx
  This is your master dataset for all analysis going forward.


In [ ]:
import pandas as pd

df = pd.read_excel('/content/Final_Clean_Dataset.xlsx')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8408 entries, 0 to 8407
Data columns (total 42 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ID_ISO                     8408 non-null   object 
 1   ID_Country                 8408 non-null   object 
 2   ID_Year                    8408 non-null   int64  
 3   ID_Region                  8408 non-null   int64  
 4   INC_Status                 8408 non-null   int64  
 5   INC_Historical             8408 non-null   int64  
 6   REV_Total_Inc_Grants       8408 non-null   float64
 7   REV_Total_Exc_Grants       8408 non-null   float64
 8   REV_Resource_Total         8408 non-null   float64
 9   REV_NonResource_Total      8408 non-null   float64
 10  TAX_Total_Inc_SC           8408 non-null   float64
 11  TAX_Total_Exc_SC           8408 non-null   float64
 12  TAX_Resource_Total         8408 non-null   float64
 13  TAX_NonResource_Total      8408 non-null   float